# ETLegacy Core Infrastructure

Generate a small Einstein Toolkit-style thorn and inspect its CCL and source files.

Navigation: [Index](../index.ipynb) | Previous: [BHaH Project Anatomy](bhah_project_anatomy.ipynb) | Next: [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)


## Learning Goals

- Create a small Einstein Toolkit-style thorn from NRPy ingredients.
- Read CCL files as declarations of variables, parameters, schedule, and source files.
- Connect scheduled functions to the time-step workflow.

## Words for This Notebook

- **Thorn:** an Einstein Toolkit component: a folder with declared variables, parameters, schedule, and source files.
- **CCL file:** a text file used by the Einstein Toolkit to describe a thorn.
- **Schedule:** the order and time when functions run.
- **Hook:** a function placed into a larger workflow at a named point.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## Generate a Small Thorn
CCL files declare thorn variables, parameters, schedule order, and source-file lists. The source function here is a compact right-hand-side example.


## Import File and Output Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [1]:
from pathlib import Path
import contextlib
import io
import re
import tempfile


## Import ETLegacy Registration Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [2]:
import nrpy.c_function as cfc
import nrpy.grid as gri
import nrpy.params as par
from nrpy.infrastructures.ETLegacy import (
    CodeParameters,
    MoL_registration,
    Symmetry_registration,
    boundary_conditions,
    interface_ccl,
    make_code_defn,
    param_ccl,
    schedule_ccl,
    simple_loop,
    zero_rhss,
)


## Create a Thorn Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [3]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_etlegacy_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / "toy_etlegacy"
THORN_NAME = "ToyETLegacyNRPy"


## Step 4: Register gridfunctions

The registry records named grid fields and their roles in generated code.

In [4]:
for name in [
    f"{THORN_NAME}_rhs_eval",
    f"{THORN_NAME}_zero_rhss",
    f"{THORN_NAME}_MoL_registration",
    f"{THORN_NAME}_Symmetry_registration_oldCartGrid3D",
    f"{THORN_NAME}_specify_Driver_BoundaryConditions",
]:
    cfc.CFunction_dict.pop(name, None)
for name in ["psi", "energy", "psi_rhs", "energy_rhs"]:
    gri.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "ETLegacy")
par.register_CodeParameter("CCTK_INT", __name__, "fd_order", 4)
par.register_CodeParameter("CCTK_REAL", __name__, "wave_speed", 1.0)
gri.register_gridfunctions(["psi"], group="EVOL")
gri.register_gridfunctions(["energy"], group="AUX")


[energy]

## Step 5: Build the Right-Hand-Side Function Body

This body combines ETLegacy parameter reads with a loop over grid points.


In [5]:
body = f"DECLARE_CCTK_ARGUMENTS_{THORN_NAME}_rhs_eval;\nDECLARE_CCTK_PARAMETERS;\n"
body += CodeParameters.read_CodeParameters(
    list_of_tuples__thorn_CodeParameter=[(THORN_NAME, "wave_speed")],
    declare_invdxxs=True,
)
body += simple_loop.simple_loop(
    loop_body=(
        f"{gri.ETLegacyGridFunction.access_gf('psi_rhs')} = "
        f"wave_speed * ({gri.ETLegacyGridFunction.access_gf('energy')} - "
        f"{gri.ETLegacyGridFunction.access_gf('psi')});"
    ),
    loop_region="interior",
)


## Step 6: Register the right-hand-side C function and its schedule entry

Register the right-hand-side C function and its schedule entry.


In [6]:
cfc.register_CFunction(
    subdirectory=THORN_NAME,
    includes=["cctk.h", "cctk_Arguments.h", "cctk_Parameters.h"],
    desc="Evaluate a toy right-hand side.",
    cfunc_type="void",
    name=f"{THORN_NAME}_rhs_eval",
    params="CCTK_ARGUMENTS",
    body=body,
    ET_thorn_name=THORN_NAME,
    ET_schedule_bins_entries=[
        (
            "MoL_CalcRHS",
            """
schedule FUNC_NAME in MoL_CalcRHS
{
  LANG: C
  READS: evol_variables(everywhere)
  READS: aux_variables(everywhere)
  WRITES: evol_variables_rhs(interior)
} "Evaluate toy right-hand side"
""",
        )
    ],
    ET_current_thorn_CodeParams_used=["wave_speed"],
)
print(cfc.CFunction_dict[f"{THORN_NAME}_rhs_eval"].function_prototype)


void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS);


## Step 7: Register zero-right-hand-side, MoL, symmetry, and boundary hooks

Register zero-right-hand-side, MoL, symmetry, and boundary hooks.


In [7]:
zero_rhss.register_CFunction_zero_rhss(THORN_NAME)
MoL_registration.register_CFunction_MoL_registration(THORN_NAME)
Symmetry_registration.register_CFunction_Symmetry_registration_oldCartGrid3D(THORN_NAME)
boundary_conditions.register_CFunction_specify_Driver_BoundaryConditions(
    thorn_name=THORN_NAME
)
print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(name)


registered C functions:
ToyETLegacyNRPy_MoL_registration
ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D
ToyETLegacyNRPy_rhs_eval
ToyETLegacyNRPy_specify_Driver_BoundaryConditions
ToyETLegacyNRPy_zero_rhss


## Step 8: List registered C functions

NRPy's stored function list now contains the right-hand-side and workflow hooks that will be written to source files.


In [8]:
print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(name)


registered C functions:
ToyETLegacyNRPy_MoL_registration
ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D
ToyETLegacyNRPy_rhs_eval
ToyETLegacyNRPy_specify_Driver_BoundaryConditions
ToyETLegacyNRPy_zero_rhss


## Step 9: Write ETLegacy thorn files

The constructors write the CCL declarations and generated source files for the thorn.

In [9]:
generation_output = io.StringIO()
with contextlib.redirect_stdout(generation_output):
    interface_ccl.construct_interface_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        inherits="Boundary Grid MethodofLines",
        USES_INCLUDEs="USES INCLUDE: Symmetry.h\nUSES INCLUDE: Boundary.h\n",
        is_evol_thorn=True,
    )
    param_ccl.construct_param_ccl(project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME)
    schedule_ccl.construct_schedule_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        STORAGE="""
STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]
""",
    )
    make_code_defn.output_CFunctions_and_construct_make_code_defn(
        project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME
    )


## Step 10: Read generator messages

The cleaned messages confirm which files the infrastructure writer produced.

In [10]:
cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", generation_output.getvalue())
cleaned = cleaned.replace(str(WORKSPACE), "<workspace>").replace(
    str(PROJECT_DIR), "project/toy_etlegacy"
)
print("generation output:")
for line in cleaned.splitlines():
    if line.strip():
        print(line.rstrip())


generation output:
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/interface.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/param.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/schedule.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_zero_rhss.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/make.code.defn...[written]


## Step 11: Check required files

The guard stops the notebook with a clear missing-file error if generation did not produce the expected artifacts.

In [11]:
required_files = [
    "interface.ccl",
    "param.ccl",
    "schedule.ccl",
    "src/make.code.defn",
    f"src/{THORN_NAME}_rhs_eval.c",
]
for relative_path in required_files:
    path = PROJECT_DIR / THORN_NAME / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)


## Step 12: Inspect the generated inventory

The inventory identifies the generated files relevant to this lesson.

In [12]:
print("thorn inventory:")
for path in sorted((PROJECT_DIR / THORN_NAME).rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR / THORN_NAME))


thorn inventory:
interface.ccl
param.ccl
schedule.ccl
src/ToyETLegacyNRPy_MoL_registration.c
src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c
src/ToyETLegacyNRPy_rhs_eval.c
src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c
src/ToyETLegacyNRPy_zero_rhss.c
src/make.code.defn


## Step 13: Read `interface.ccl`

The `interface.ccl` file declares grid variables and inherited capabilities.


In [13]:
print(
    (PROJECT_DIR / THORN_NAME / "interface.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)



# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: ToyETLegacyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Boundary Grid MethodofLines

# Needed functions and #include's:
USES INCLUDE: Symmetry.h
USES INCLUDE: Boundary.h


# Needed Method of Lines function
CCTK_INT FUNCTION MoLRegisterEvolvedGroup(CCTK_INT IN EvolvedIndex, CCTK_INT IN RHSIndex)
REQUIRES FUNCTION MoLRegisterEvolvedGroup

# Needed Boundary Conditions function
CCTK_INT FUNCTION GetBoundarySpecification(CCTK_INT IN size, CCTK_INT OUT ARRAY nboundaryzones, CCTK_INT OUT ARRAY is_internal, CCTK_INT OUT ARRAY is_staggered, CCTK_INT OUT ARRAY shiftout)
USES FUNCTION GetBoundarySpecification

CCTK_INT FUNCTION SymmetryTableHandleForGrid(CCTK

## Step 14: Read `param.ccl`

The parameter file declares runtime parameters exposed by the thorn.

In [14]:
print(
    (PROJECT_DIR / THORN_NAME / "param.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)


# This param.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.


restricted:
CCTK_INT fd_order "(see NRPy for parameter definition)"
{
 *:* :: "All values accepted. NRPy does not restrict the allowed ranges of parameters yet."
} 4

CCTK_REAL wave_speed "(see NRPy for parameter definition)"
{
 *:* :: "All values accepted. NRPy does not restrict the allowed ranges of parameters yet."
} 1.0




## Step 15: Read `schedule.ccl`

The schedule file places functions in Einstein Toolkit schedule bins.

In [15]:
print(
    (PROJECT_DIR / THORN_NAME / "schedule.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)


# This schedule.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

##################################################
# Step 0: Allocate memory for gridfunctions, using the STORAGE: keyword.

STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]


##################################################
# Step 1: Schedule functions in the Driver_BoundarySelect scheduling bin.

schedule ToyETLegacyNRPy_specify_Driver_BoundaryConditions in Driver_BoundarySelect
{
  LANG: C
  OPTIONS: meta
} "Register boundary conditions in PreSync bin Driver_BoundarySelect."

##################################################
# Step 2: Schedule functions in the BASEGRID scheduling bin.

schedule ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D at BASEGRID as Symmetry_registration
{
  LANG: C
  OPTIONS: Global
} "Register symmetries, the CartGrid3D way."

schedule ToyETLegacyNRPy

## Step 16: Read `make.code.defn`

This file lists source files compiled into the thorn.

In [16]:
print(
    (PROJECT_DIR / THORN_NAME / "src" / "make.code.defn").read_text(
        encoding="utf-8", errors="replace"
    )
)


# make.code.defn file for thorn ToyETLegacyNRPy

# Source files that need to be compiled:
SRCS = \
       ToyETLegacyNRPy_MoL_registration.c \
       ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c \
       ToyETLegacyNRPy_rhs_eval.c \
       ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c \
       ToyETLegacyNRPy_zero_rhss.c



## Step 17: Read the Right-Hand-Side Source

The source file is the generated C function registered earlier. The full file is printed because this notebook is teaching how an Einstein Toolkit-style thorn is assembled. Look for the includes, the scheduled function name, the parameter read, the grid loop, and the assignment that evaluates the toy right-hand side.


In [17]:
print(
    (PROJECT_DIR / THORN_NAME / "src" / f"{THORN_NAME}_rhs_eval.c").read_text(
        encoding="utf-8", errors="replace"
    )
)


#include "cctk.h"
#include "cctk_Arguments.h"
#include "cctk_Parameters.h"

/**
 * Evaluate a toy right-hand side.
 */
void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS) {
  DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval;
  DECLARE_CCTK_PARAMETERS;
  const CCTK_REAL *wave_speedptr = CCTK_ParameterGet("wave_speed", "ToyETLegacyNRPy", NULL); // ToyETLegacyNRPy::wave_speed
  const CCTK_REAL wave_speed = *wave_speedptr;
  const CCTK_REAL invdxx0 = 1.0 / CCTK_DELTA_SPACE(0);
  const CCTK_REAL invdxx1 = 1.0 / CCTK_DELTA_SPACE(1);
  const CCTK_REAL invdxx2 = 1.0 / CCTK_DELTA_SPACE(2);
#pragma omp parallel for
  for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2] - cctk_nghostzones[2]; i2++) {
    for (int i1 = cctk_nghostzones[1]; i1 < cctk_lsh[1] - cctk_nghostzones[1]; i1++) {
      for (int i0 = cctk_nghostzones[0]; i0 < cctk_lsh[0] - cctk_nghostzones[0]; i0++) {
        psi_rhsGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] =
            wave_speed * (energyGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] - psiG

The printed CCL and source files show how the same registered C-function idea is expressed as Einstein Toolkit thorn metadata. The schedule entries state when each generated function runs.


## Learning Check

Before reading the CCL files, predict which file should describe variables, which should describe parameters, and which should describe when functions run.


## Continue to Carpet WaveToy Thorns
- [C Function Registration](../1-intro/c_function.ipynb)
- [BHaH Project Anatomy](bhah_project_anatomy.ipynb)
- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)
- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)
